# Data Mining - Lab 8
## Data Cube ба Attribute-Oriented Induction (AOI)

**Dataset:** `cube_small.csv`, `cube_dim_month.csv`, `cube_dim_product.csv`, `cube_dim_branch.csv`, `cube_fact_sales.csv`, `cube_fact_inventory.csv`, `cube_students_aoi.csv`, `cube_payments_flat.csv`

Энэ лаб нь "Data Cube: Атрибутэд суурилсан индукцлэл" лекцийн жишээнүүд дээр үндэслэсэн. Бичиг хэргийн сүлжээ дэлгүүрийн борлуулалтын cube, оюутнуудын AOI жишээ, сургалтын төлбөрийн схем гурван хэсгээс бүрдэнэ.

| Файл | Мөр | Тайлбар |
|---|---|---|
| `cube_small` | 8 | Лекцийн 2 сар x 2 бараа x 2 салбарын cube (хэмжүүр: орлого, мянган ₮) |
| `cube_fact_sales` | 864 | Од схемийн fact: сар x бараа x салбар (2025-2026 он), `revenue`, `quantity` |
| `cube_dim_month`, `cube_dim_product`, `cube_dim_branch` | 24, 6, 6 | Од схемийн dimension хүснэгтүүд |
| `cube_fact_inventory` | 1116 | 2026 оны 1-р сарын өдөр бүрийн үлдэгдэл (snapshot fact) |
| `cube_students_aoi` | 150 | AOI-д зориулсан оюутны өгөгдөл (`S1`-`S6` нь лекцийн жишээ) |
| `cube_payments_flat` | 467 | Сургалтын төлбөрийн задлаагүй хүснэгт (схем зохиох даалгаварт) |

### Lab зорилго
- Dimension, measure, cell, cuboid ойлголтыг кодоор гаргаж, 2ⁿ cuboid-ийг тооцох
- Roll-up, drill-down, slice, dice, pivot үйлдлийг pandas-аар хийх
- Distributive, algebraic, holistic хэмжүүрийн нэгтгэлийн дүрмийг шалгах
- AOI-ийн алхмаар (атрибут сонгох, ерөнхийлөх, нэгтгэх, босго шалгах) ерөнхийлсөн хүснэгт гаргах
- Од (star), цасан ширхэг (snowflake), fact constellation схемийг хэрэгжүүлж, чанарын шалгалт хийх

Даалгавар бүр **TASK** гэсэн гарчигтай. `___` гэсэн хоосон хэсгийг бөглөнө. Хүлээгдэх хариуг даалгаврын тайлбарт өгсөн.

## 0) Өгөгдөл унших

Эхлээд лекцийн жижиг cube-ийг уншина. Dimension нь `month`, `product`, `branch`, хэмжүүр нь `revenue` (мянган ₮).

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

small = pd.read_csv("cube_small.csv")
print(small.shape)
small

(8, 4)


,month,product,branch,revenue
0,2026-01,Ном,УБ,120
1,2026-01,Ном,Дархан,60
2,2026-01,Дэвтэр,УБ,80
3,2026-01,Дэвтэр,Дархан,40
4,2026-02,Ном,УБ,150
5,2026-02,Ном,Дархан,70
6,2026-02,Дэвтэр,УБ,90
7,2026-02,Дэвтэр,Дархан,50


## 1) Data Cube-ийн үндсэн ойлголт

- **Dimension**: ямар өнцгөөс шинжлэх вэ (сар, бараа, салбар)
- **Measure**: юуг тооцох вэ (орлого)
- **Cell**: хэмжээсийн нэг хослолын хэмжүүрийн утга. Жишээ нь (2026-01, Ном, УБ) = 120

### TASK 1: Cube-ийн нэгтгэл (лекцийн Бодлого 1)

1. Cube-д хэдэн cell байна вэ? Dimension бүрийн ялгаатай утгын тоог үржүүлж баталгаажуул.
2. УБ салбарын хоёр сарын нийт орлогыг ол.
3. Номын бүх салбарын хоёр сарын нийт орлогыг ол.
4. Ном, УБ, 1-2-р сар гэсэн шүүлтийн орлогыг ол.
5. Нэгтгэхдээ **үлдээх хэмжээс** ба **шүүх нөхцөл**-ийн ялгааг өөрийн үгээр тайлбарла.
8 cell, нийт 660, УБ = 440, Ном = 400, шүүлт = 270. Яг л энэ нь “үлдээх хэмжээс” (remaining dimensions) болон “шүүх нөхцөл” (filter condition)-ийн ялгааг харуулж байна.


Хүлээгдэх хариу: 8 cell, нийт 660, УБ = 440, Ном = 400, шүүлт = 270.

In [2]:
dims = ["month", "product", "branch"]
n_cells = 1
for c in dims:
    n_cells *= small[c].nunique()
print("cell:", n_cells, "нийт:", small["revenue"].sum())

ub   = small[small["branch"] == "УБ"]["revenue"].sum()
book = small[small["product"] == "Ном"]["revenue"].sum()
dice = small[(small["product"] == "Ном") & (small["branch"] == "УБ")]["revenue"].sum()
print("УБ:", ub, "Ном:", book, "Ном+УБ:", dice)

cell: 8 нийт: 660
УБ: 440 Ном: 400 Ном+УБ: 270


## 2) Cuboid ба нэгтгэлийн түвшин

n хэмжээстэй cube-д dimension-ы дэд олонлог бүр нэг cuboid болно, нийт **2ⁿ**. 3 хэмжээсийн хувьд 8: 1 base (3D), 3 ширхэг 2D, 3 ширхэг 1D, 1 apex (0D, ALL).

### TASK 2: 8 cuboid-ийг гаргах

1. `combinations`-ээр dimension-ы бүх дэд олонлогийг (0-ээс 3 хэмжээс) гарга.
2. Cuboid бүрийн cell-ийн тоо ба `revenue`-ийн нийлбэрийг хэвлэ.
3. Бүх cuboid-ийн нийлбэр ижил байх ёстой. Яагаад?
3 хэмжээст cube-д нийт 8 cuboid бий. Base cuboid-д 8 cell, apex cuboid-д 1 cell. Бүх cuboid-ийн нийлбэр ижил 660 байгаа нь бүгд нэг ижил fact-ийг different aggregation level-ээр харж байна.

4. Base ба apex cuboid-ийг ялга.

Хүлээгдэх хариу: 8 cuboid. Cell-ийн тоо: 8 (base), 4, 4, 4, 2, 2, 2, 1 (apex). Нийлбэр бүгд 660.

In [3]:
rows = []
for k in range(len(dims), -1, -1):
    for sub in combinations(dims, k):
        if sub:
            g = small.groupby(list(sub))["revenue"].sum()
            rows.append((sub, len(g), g.sum()))
        else:
            rows.append((("ALL",), 1, small["revenue"].sum()))

for sub, ncell, tot in rows:
    print(f"{str(sub):45s} cell={ncell}  sum={tot}")
print("cuboid:", len(rows))

('month', 'product', 'branch')                cell=8  sum=660
('month', 'product')                          cell=4  sum=660
('month', 'branch')                           cell=4  sum=660
('product', 'branch')                         cell=4  sum=660
('month',)                                    cell=2  sum=660
('product',)                                  cell=2  sum=660
('branch',)                                   cell=2  sum=660
('ALL',)                                      cell=1  sum=660
cuboid: 8


### TASK 3: Шатлалтай cube-ийн cuboid-ийн тоо

Dimension бүр шатлалтай бол cuboid-ийн тоо **T = (L₁+1)(L₂+1)...(Lₙ+1)**, энд Lᵢ нь ALL-ийг оруулаагүй түвшний тоо.

| Dimension | Шатлал | Түвшин |
|---|---|---|
| Хугацаа | сар, улирал, жил | 3 |
| Бараа | бараа, ангилал | 2 |
| Салбар | салбар, хот, бүс | 3 |

1. Нийт cuboid-ийн тоог бод. 48
2. Хугацаанд `өдөр` түвшин нэмбэл хэд болох вэ? 60
3. Яагаад шатлал нэмэхэд тоо хурдан өсдөг вэ? Бүгдийг урьдчилан тооцох (materialize) нь ямар зардалтай вэ?
Шатлал нэмэх тутам тоо хурдан нэмэгддэг, учир нь түүний үржвэр хэлбэрээр тооцогддог.

Хүлээгдэх хариу: 48 ба 60.

In [4]:
levels = {"time": 3, "product": 2, "branch": 3}
total = 1
for L in levels.values():
    total *= (L + 1)
print("нийт cuboid:", total)
print("өдөр нэмбэл:", np.prod([L + 1 for L in dict(levels, time=4).values()]))

нийт cuboid: 48
өдөр нэмбэл: 60


Хариулт: 48, өдөр нэмбэл 60. Түвшний тоо үржигдэж байгаа учраас хурдан өснө. Бүгдийг урьдчилан тооцвол санах ой их шаардана, өгөгдөл өөрчлөгдөх бүрт дахин бодох хэрэгтэй.

## 3) Од схем: fact ба dimension холбох

Одоо 2025-2026 оны бүтэн cube-ийг од схемээс уншина. **Grain**: нэг мөр = нэг сар x нэг бараа x нэг салбар.

In [5]:
fact  = pd.read_csv("cube_fact_sales.csv")
d_mon = pd.read_csv("cube_dim_month.csv")
d_pro = pd.read_csv("cube_dim_product.csv")
d_bra = pd.read_csv("cube_dim_branch.csv")
for n, t in [("fact", fact), ("dim_month", d_mon), ("dim_product", d_pro), ("dim_branch", d_bra)]:
    print(n, t.shape)
fact.head()

fact (864, 6)
dim_month (24, 6)
dim_product (6, 4)
dim_branch (6, 4)


,sales_key,month_key,product_key,branch_key,revenue,quantity
0,1,1,1,1,100,10
1,2,1,1,2,120,12
2,3,1,1,3,10,1
3,4,1,1,4,130,13
4,5,1,1,5,70,7


### TASK 4: Холболт ба нийлбэрийн тулгалт

1. Fact-ийн гурван foreign key бүр dimension-д байгааг шалга. Тус бүр `True`.
2. Dimension-уудыг `validate="many_to_one"`-тэй холбож `dw` гарга.
3. Холболтын дараа мөрийн тоо ба `revenue`-ийн нийлбэр өөрчлөгдөөгүй байх ёстой.
4. Grain шалга: (month_key, product_key, branch_key) хослол давхцаагүй эсэхийг `duplicated()`-аар шалга.

Fact-ийн 3 foreign key бүгд dimension-д байна. Холболт хийсний дараа 864 мөр, нийт орлого 36449, давхардал 0 байна.


Хүлээгдэх хариу: 864 мөр, нийт орлого 36449, давхардал 0.

In [6]:
print(fact["month_key"].isin(d_mon["month_key"]).all())
print(fact["product_key"].isin(d_pro["product_key"]).all())
print(fact["branch_key"].isin(d_bra["branch_key"]).all())

dw = (fact
      .merge(d_mon, on="month_key", validate="many_to_one")
      .merge(d_pro, on="product_key", validate="many_to_one")
      .merge(d_bra, on="branch_key", validate="many_to_one"))

print(len(dw) == len(fact), dw["revenue"].sum() == fact["revenue"].sum(), dw["revenue"].sum())
dup = fact.duplicated(subset=["month_key", "product_key", "branch_key"]).sum()
print("grain давхардал:", dup)

True
True
True
True True 36449
grain давхардал: 0


## 4) Roll-up ба drill-down

Шатлал:

- Хугацаа: **сар -> улирал -> жил**
- Салбар: **салбар -> хот -> бүс**

**Roll-up** нь илүү ерөнхий түвшинд нэгтгэнэ (бүлэглэх багана багасна). **Drill-down** нь дэлгэрэнгүй түвшинд задална.

### TASK 5: Хугацаа ба байршлын шатлал

1. 2026 оны орлогыг **жилээр**, дараа нь **улирлаар** гарга. Улирлын нийлбэр жилийнхтэй тэнцэх ёстой.
2026 оны нийт орлого 18970. Улиралын хувьд Q1 = 4908, Q2 = 4367, Q3 = 4552, Q4 = 5143.
2. Хамгийн их орлоготой улирал руу drill-down хийж **сараар** задал.
3. 2026 оны орлогыг **бүс**, дараа нь **хот**, дараа нь **салбараар** гарга.
4. Улаанбаатар хотын орлого хэдэн салбараас бүрдэж байна вэ? Хотын түвшинд roll-up хийхэд яагаад `УБ` ба `УБ-2` нэг мөр болдог вэ?

 Хамгийн их нь Q4, доторх 2026-10 сар 1846. Бүсийн хувьд Төвийн бүс хамгийн их 9461.


Хүлээгдэх хариу: 2026 жил = 18970 (2025 = 17479). Улирал: Q1 4908, Q2 4367, Q3 4552, Q4 5143. Хамгийн их нь Q4, түүний дотор 2026-10 сар (1846). Бүс: Төв нэгдүгээрт (9461, нийт 49.9%). Улаанбаатар 9461.

In [7]:
d26 = dw[dw["year"] == 2026]

# roll-up: сар -> улирал -> жил
by_year = d26.groupby("year")["revenue"].sum()
by_q    = d26.groupby("quarter")["revenue"].sum()
print(by_year.to_dict()); print(by_q.to_dict())
print("тэнцэх үү:", by_q.sum() == by_year.iloc[0])

# drill-down: хамгийн их улирлыг сараар задлах
top_q = by_q.idxmax()
print(top_q, d26[d26["quarter"] == top_q].groupby("month")["revenue"].sum().to_dict())

# салбарын шатлал
print(d26.groupby("region")["revenue"].sum().sort_values(ascending=False).to_dict())
print(d26.groupby("city")["revenue"].sum().sort_values(ascending=False).to_dict())
print(d26.groupby(["city", "branch_name"])["revenue"].sum())

{2026: 18970}
{'Q1': 4908, 'Q2': 4367, 'Q3': 4552, 'Q4': 5143}
тэнцэх үү: True
Q4 {'2026-10': 1846, '2026-11': 1632, '2026-12': 1665}
{'Төв': 9461, 'Хойд': 5419, 'Баруун': 2236, 'Хангай': 1854}
{'Улаанбаатар': 9461, 'Дархан': 3162, 'Эрдэнэт': 2257, 'Ховд': 2236, 'Мөрөн': 1854}
city         branch_name
Дархан       Дархан         3162
Мөрөн        Мөрөн          1854
Улаанбаатар  УБ             5522
             УБ-2           3939
Ховд         Ховд           2236
Эрдэнэт      Эрдэнэт        2257
Name: revenue, dtype: int64


Хариулт: 2026 он 18970, улирлын нийлбэр мөн 18970. Q4 хамгийн их, дотроо 10 сар 1846. Улаанбаатар 2 салбартай (УБ 5522, УБ-2 3939). Хотоор бүлэглэхэд city нь адилхан учир нэг мөр болно.

## 5) Slice, dice ба pivot

| Үйлдэл | Юу хийх вэ | Жишээ |
|---|---|---|
| Slice | Нэг хэмжээсийн нэг утгыг тогтооно | `month == "2026-01"` |
| Dice | Хэд хэдэн хэмжээсээр дэд муж сонгоно | сар in {1, 2}, бараа in {Ном, Дэвтэр}, салбар in {УБ, Дархан} |
| Pivot | Мөр, баганыг солино (утга өөрчлөгдөхгүй) | бараа багана, салбар мөр |

### TASK 6: Slice, dice, pivot

1. **Slice**: 2026-01 сарын нийт орлого, түүнээс хамгийн их борлуулалттай бараа.
- 026-01 сарын slice-ийн нийт орлого 1380, хамгийн их борлуулалттай бараа нь Үүргэвч (480)
2. **Dice**: лекцийн нөхцөлөөр (2026-01 ба 2026-02, Ном ба Дэвтэр, УБ ба Дархан) мөрийн тоо, нийлбэр.
3. **Pivot**: dice-ийн үр дүнг `index = салбар`, `columns = бараа` болгож, дараа нь `index` ба `columns`-ыг солиж хар.
4. Pivot-ийн хоёр хэлбэрийн нийт нийлбэр ижил гэдгийг батал.
- Dice-ээс 8 мөр, нийт 660. Pivot хийсэн ч нийт нийлбэр өөрчлөгддөггүй.


Хүлээгдэх хариу: slice нийт 1380, хамгийн их бараа Үүргэвч (480). Dice 8 мөр, нийлбэр 660 (лекцийн cube-тэй ижил 120 + 60 + 80 + 40 + 150 + 70 + 90 + 50 = 660). Нийт нийлбэр 660.

In [8]:
# 1. slice
sl = dw[dw["month"] == "2026-01"]
print(sl["revenue"].sum(), sl.groupby("product_name")["revenue"].sum().idxmax())

# 2. dice
dice = dw[dw["month"].isin(["2026-01", "2026-02"])
          & dw["product_name"].isin(["Ном", "Дэвтэр"])
          & dw["branch_name"].isin(["УБ", "Дархан"])]
print(len(dice), dice["revenue"].sum())

# 3. pivot
pv = dice.pivot_table(index="branch_name", columns="product_name", values="revenue", aggfunc="sum")
print(pv)
print(pv.T)
print(pv.values.sum() == pv.T.values.sum())

1380 Үүргэвч
8 660
product_name  Дэвтэр  Ном
branch_name              
Дархан            90  130
УБ               170  270
branch_name   Дархан   УБ
product_name             
Дэвтэр            90  170
Ном              130  270
True


## 6) Хэмжүүр ба нэгтгэлийн дүрэм

| Ангилал | Жишээ | Хэсгийн үр дүнгээс нийт тооцох |
|---|---|---|
| Distributive | SUM, COUNT, MIN, MAX | Хэсгийн дүнгээр шууд (COUNT-ийн дүнг SUM хийнэ) |
| Algebraic | AVG | Хэсэг бүрийн **SUM ба COUNT**-ыг хадгална |
| Holistic | MEDIAN | Тогтмол хэмжээтэй хураангуйгаар тооцох боломжгүй |

### TASK 7: Дундажийн дундаж алдаа

1. Лекцийн жишээ: А бүлэг SUM = 100, COUNT = 10; Б бүлэг SUM = 90, COUNT = 3. Зөв нийт AVG ба "дундажийн дундаж"-ыг бод.
2. Бодит өгөгдөл: `revenue`-ийн дундажийг **зөв** (бүх мөрөөр) ба **буруу** (бүс бүрийн дундажийн дундаж) аргаар бод. Яагаад зөрж байна вэ?
3. Зөв аргыг бүс бүрийн `SUM` ба `COUNT`-оор (algebraic) дахин гарга.
4. Median-ыг мөн адил (бүсийн median-уудын median) бод. Алдаа гарах уу? Яагаад?
- Зөв дундаж 14.62, буруу “average of averages” 20.0. Бодит өгөгдлийн дундаж 42.19, буруу арга 38.63. Энэ нь group-үүдийн тоо болон бүртгэлүүдийн хэмжээнээс болоод зөрдөг. Median-ийг зөвөөр тооцвол 20.0, буруу аргаар 13.25. Тиймээс algebraic measure-ийг SUM + COUNT-оор тооцох нь зөв.

Хүлээгдэх хариу: 14.62 ба 20.0. Бодит дундаж 42.19 ба буруу 38.63. Median зөв 20.0, буруу 13.25.

In [9]:
# 1. лекцийн жишээ
sums, cnts = np.array([100, 90]), np.array([10, 3])
print("зөв:", round(sums.sum() / cnts.sum(), 2), " буруу:", (sums / cnts).mean())

# 2. бодит өгөгдөл
true_avg  = dw["revenue"].mean()
wrong_avg = dw.groupby("region")["revenue"].mean().mean()
print(round(true_avg, 2), round(wrong_avg, 2))

# 3. algebraic: SUM ба COUNT хадгална
part = dw.groupby("region")["revenue"].agg(["sum", "count"])
print(round(part["sum"].sum() / part["count"].sum(), 2))

# 4. holistic
print(dw["revenue"].median(), dw.groupby("region")["revenue"].median().median())

зөв: 14.62  буруу: 20.0
42.19 38.63
42.19
20.0 13.25


## 7) Attribute-Oriented Induction (AOI)

AOI нь нарийвчилсан атрибутуудыг **шатлалаар ерөнхийлж**, ижил болсон мөрүүдийг нэгтгэн, `count`-оор бүлгийн шинжийг тайлбарлана.

| Алхам | Хийх ажил |
|---|---|
| 1. Өгөгдөл сонгох | Зорилтот бүлэг, хэрэгтэй атрибутууд |
| 2. Ерөнхийлөх | Утгыг дээд ойлголтод шилжүүлэх, хэрэггүй атрибутыг хасах |
| 3. Нэгтгэх | Ижил мөрүүдийг нэгтгэж `count` нэмэх |
| 4. Хянах | Атрибутын утгын тоо ба үр дүнгийн мөрийн босго |

### TASK 8: Лекцийн 6 оюутан

Шатлал: нас `20-29` ба `30-39`; `CS`, `SE` -> `Мэдээллийн технологи`; `Нягтлан`, `Санхүү` -> `Бизнес`.

1. `student_id`-г яагаад ерөнхийлөхгүй, харин **хасдаг** вэ? (үгээр)
2. Нас ба мэргэжлийг ерөнхийлж, ижил мөрүүдийг нэгтгэж `count` ба эзлэх хувийг тооц.
3. `count`-уудын нийлбэр эх мөрийн тоотой (6) тэнцэх ёстой. Шалга.
4. "66.7% нь IT чиглэлийн 20-29 настай" гэсэн дүгнэлт шалтгаан нотлох уу? (үгээр)

Хүлээгдэх хариу: (20-29, Мэдээллийн технологи) = 4 (66.7%); (30-39, Бизнес) = 2 (33.3%).

In [10]:
stu = pd.read_csv("cube_students_aoi.csv")
six = stu[stu["student_id"].isin(["S1", "S2", "S3", "S4", "S5", "S6"])].copy()

def age_group(a):
    return "20-29" if a < 30 else "30-39"

major_map = {"CS": "Мэдээллийн технологи", "SE": "Мэдээллийн технологи",
             "Нягтлан": "Бизнес", "Санхүү": "Бизнес"}

six["age_g"]   = six["age"].apply(age_group)
six["major_g"] = six["major"].map(major_map)

res = six.groupby(["age_g", "major_g"]).size().reset_index(name="count")
res["percent"] = (res["count"] / res["count"].sum() * 100).round(1)
print(res)
print("нийлбэр:", res["count"].sum())

   age_g               major_g  count  percent
0  20-29  Мэдээллийн технологи      4     66.7
1  30-39                Бизнес      2     33.3
нийлбэр: 6


Хариулт:
1. student_id бүгд өөр утгатай, ерөнхийлөх шатлалгүй учир хасна. Үлдээвэл нэг ч мөр нийлэхгүй.
2-3. (20-29, МТ) 4 буюу 66.7%, (30-39, Бизнес) 2 буюу 33.3%, нийлбэр 6.
4. Үгүй, энэ зөвхөн өгөгдлийг дүрсэлж байгаа болохоос шалтгааныг батлахгүй. Түүвэр ч жижиг.

### TASK 9: 150 оюутны AOI

Одоо бүх 150 оюутан дээр AOI хий.

**Алхам 1: Атрибутын утгын тоог шалга.** `student_id` (150 өвөрмөц), `age` (25), `major` (11), `gpa` (19), `city` (6). Өвөрмөц утга олонтой атрибутыг ерөнхийлөх шатлалтай эсэхээр нь шийд.

**Алхам 2: Шатлал.**

| Атрибут | Ерөнхийлөлт |
|---|---|
| `age` | 20-29, 30-39, 40-49 |
| `major` | CS, SE, DS, IS -> Мэдээллийн технологи; Нягтлан, Санхүү, Маркетинг, Менежмент -> Бизнес; бусад -> Инженер |
| `gpa` | [0, 2.5) Хангалттай; [2.5, 3.5) Сайн; [3.5, 4.0] Онц |
| `student_id` | хасна (ерөнхийлөх шатлалгүй) |

**Алхам 3: Нэгтгэ.** `age_g` ба `major_g`-ээр бүлэглэж `count`, эзлэх хувийг гарга. Хувиар буурахаар эрэмбэл.

**Алхам 4: Босго.** Ерөнхийлөлтөд `gpa_g`-г нэмбэл мөрийн тоо хэд болох вэ? Мөрийн босго 10 гэж үзвэл аль хувилбар тохирох вэ?

Хүлээгдэх хариу: `age_g` + `major_g` = 9 мөр, хамгийн том нь (20-29, Мэдээллийн технологи) = 63 оюутан (42.0%). Эхний хоёр мөр нийт 61.3%. `gpa_g` нэмбэл 20 мөр.

In [11]:
print({c: stu[c].nunique() for c in ["student_id", "age", "major", "gpa", "city"]})

it  = ["CS", "SE", "DS", "IS"]
biz = ["Нягтлан", "Санхүү", "Маркетинг", "Менежмент"]
def major_group(x):
    if x in it:  return "Мэдээллийн технологи"
    if x in biz: return "Бизнес"
    return "Инженер"

g = stu.drop(columns=["student_id"]).copy()
g["age_g"]   = pd.cut(g["age"], bins=[19, 29, 39, 49], labels=["20-29", "30-39", "40-49"]).astype(str)
g["major_g"] = g["major"].apply(major_group)
g["gpa_g"]   = pd.cut(g["gpa"], bins=[0, 2.5, 3.5, 4.01], labels=["Хангалттай", "Сайн", "Онц"], right=False).astype(str)

aoi = (g.groupby(["age_g", "major_g"]).size().reset_index(name="count")
         .sort_values("count", ascending=False))
aoi["percent"] = (aoi["count"] / aoi["count"].sum() * 100).round(1)
print(aoi.to_string(index=False))
print("нийлбэр:", aoi["count"].sum())

aoi3 = g.groupby(["age_g", "major_g", "gpa_g"]).size()
print("gpa нэмбэл мөр:", len(aoi3))

{'student_id': 150, 'age': 25, 'major': 11, 'gpa': 19, 'city': 6}
age_g              major_g  count  percent
20-29 Мэдээллийн технологи     63     42.0
20-29               Бизнес     29     19.3
30-39               Бизнес     18     12.0
20-29              Инженер     12      8.0
40-49               Бизнес     10      6.7
30-39              Инженер      7      4.7
40-49 Мэдээллийн технологи      6      4.0
30-39 Мэдээллийн технологи      3      2.0
40-49              Инженер      2      1.3
нийлбэр: 150
gpa нэмбэл мөр: 20


Хариулт: age_g + major_g 9 мөр, хамгийн том нь (20-29, МТ) 63 оюутан (42%). gpa_g нэмбэл 20 мөр. Босго 10 бол 9 мөртэй хувилбар тохирно.

## 8) Data Cube ба AOI-ийн холбоо

Cube-ийн roll-up нь AOI-ийн ерөнхийлөлттэй адил шатлал ашиглана. Өөр нь: cube **хэмжүүр**-ийг (орлого) нэгтгэдэг, AOI **атрибутын утга**-ыг ерөнхийлж `count` гаргадаг.

### TASK 10: Cube-ийн нэгтгэлийг AOI маягаар унших

`dw`-ээс 2026 оны мөрүүдийг авч, `region` ба `category`-оор бүлэглэн **орлого** ба **мөрийн тоо (count)**-г зэрэг гарга. Дараа нь:

1. Хамгийн их орлоготой (бүс, ангилал) хослол аль вэ?

2. Орлого ба count-ын эрэмбэ яагаад зөрж болох вэ? (үгээр)

Хүлээгдэх хариу: 4 бүс x 3 ангилал = 12 мөр, нийт орлого 18970.

In [12]:
cc = (d26.groupby(["region", "category"])
         .agg(revenue=("revenue", "sum"), rows=("revenue", "count"))
         .reset_index().sort_values("revenue", ascending=False))
print(cc.head(5).to_string(index=False))
print(len(cc), cc["revenue"].sum())

region           category  revenue  rows
   Төв Сургалтын хэрэгсэл     6560    48
  Хойд Сургалтын хэрэгсэл     3790    48
   Төв           Электрон     1875    24
Баруун Сургалтын хэрэгсэл     1600    24
Хангай Сургалтын хэрэгсэл     1350    24
12 18970


Хариулт: 12 мөр, нийт 18970. Хамгийн их нь (Төв, Сургалтын хэрэгсэл) 6560. Count нь мөрийн тоо. Бичиг хэрэгт 3 бараа байгаа учир мөр олон ч үнэ нь хямд тул орлого бага.

## 9) Схем зохиох: flat хүснэгтээс од схем

`cube_payments_flat.csv` нь нэг мөр = **нэг оюутны нэг сургалтад төлсөн нэг төлбөр** (лекцийн Бодлого 3). Бүх мэдээлэл нэг хүснэгтэд давтагдан байна. Үүнийг од схем болгон хуваана:

- `dim_date`: `date_key`, `date`, `month`, `quarter`, `year`
- `dim_course`: `course_key`, `course_name`, `field`
- `dim_branch`: `branch_key`, `branch_name`, `city`
- `fact_payment`: `payment_key`, `date_key`, `course_key`, `branch_key`, `student_id`, `amount`

`student_id`-г fact-д хэвээр үлдээнэ (degenerate dimension). Grain: **нэг төлбөр**. Хэмжүүр: `SUM(amount)`.

### TASK 11: Од схемийг үүсгэж, чанарын шалгалт хий

1. Dimension бүрийг `drop_duplicates()`-ээр гаргаж, суррогат түлхүүр (`1, 2, 3...`) өг. `date_key`-г `YYYYMMDD` бүхэл тоо болго.
2. Flat хүснэгтийг dimension-уудтай холбож `fact_payment` үүсгэ.
3. Шалгалт: (a) dimension бүрийн PK давхцаагүй, (b) fact-ийн FK бүгд хүчинтэй, (c) мөрийн тоо flat-тай ижил, (d) нийлбэр flat-тай ижил, (e) `payment_id` давхцаагүй.
4. `COUNT(*)` юуг тоолж байна вэ? Төлбөрийн тоо ба өвөрмөц оюутны тоог харьцуул.

Хүлээгдэх хариу: 467 төлбөр, нийт 101775 мянган ₮, өвөрмөц оюутан 160. `COUNT(*)` нь **төлбөрийн үйл явдлын тоо**, оюутны тоо биш.

In [13]:
flat = pd.read_csv("cube_payments_flat.csv")
flat["pay_date"] = pd.to_datetime(flat["pay_date"])

dim_date = flat[["pay_date"]].drop_duplicates().sort_values("pay_date").rename(columns={"pay_date": "date"})
dim_date["date_key"] = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date["month"]    = dim_date["date"].dt.strftime("%Y-%m")
dim_date["quarter"]  = "Q" + dim_date["date"].dt.quarter.astype(str)
dim_date["year"]     = dim_date["date"].dt.year

dim_course = flat[["course_name", "field"]].drop_duplicates().reset_index(drop=True)
dim_course.insert(0, "course_key", range(1, len(dim_course) + 1))

dim_branch = flat[["branch_name", "city"]].drop_duplicates().reset_index(drop=True)
dim_branch.insert(0, "branch_key", range(1, len(dim_branch) + 1))

fact_payment = (flat
    .merge(dim_date[["date", "date_key"]], left_on="pay_date", right_on="date")
    .merge(dim_course, on=["course_name", "field"])
    .merge(dim_branch, on=["branch_name", "city"])
    [["payment_id", "date_key", "course_key", "branch_key", "student_id", "amount"]]
    .rename(columns={"payment_id": "payment_key"}))

print("PK давхцаагүй:", dim_date["date_key"].is_unique, dim_course["course_key"].is_unique, dim_branch["branch_key"].is_unique)
print("FK хүчинтэй:", fact_payment["date_key"].isin(dim_date["date_key"]).all(),
      fact_payment["course_key"].isin(dim_course["course_key"]).all(),
      fact_payment["branch_key"].isin(dim_branch["branch_key"]).all())
print("мөр:", len(fact_payment) == len(flat), " нийлбэр:", fact_payment["amount"].sum() == flat["amount"].sum(), fact_payment["amount"].sum())
print("payment_key давхцаагүй:", fact_payment["payment_key"].is_unique)
print("төлбөрийн тоо:", len(fact_payment), " өвөрмөц оюутан:", fact_payment["student_id"].nunique())

PK давхцаагүй: True True True
FK хүчинтэй: True True True
мөр: True  нийлбэр: True 101775
payment_key давхцаагүй: True
төлбөрийн тоо: 467  өвөрмөц оюутан: 160


Хариулт: Бүх шалгалт True. 467 төлбөр, 101775, өвөрмөц оюутан 160. COUNT(*) төлбөрийн тоог тоолно, нэг оюутан хэд хэдэн удаа төлсөн байж болно.

### TASK 12: Схем дээрх OLAP ба AOI (лекцийн Бодлого 3)

`fact_payment` ба dimension-уудыг холбож:

1. **Slice**: 2026 оны 3-р сарын нийт төлбөр ба төлбөрийн тоо.
2. **Roll-up**: хотын түвшинд нийт төлбөр. Хамгийн их хот аль вэ?
3. **AOI**: сургалтыг `field` (чиглэл)-ээр ерөнхийлж, төлбөрийн `count` ба нийлбэрийг гарга.

Хүлээгдэх хариу: 3-р сар 16463 (78 төлбөр). Хамгийн их хот Улаанбаатар (46885). Чиглэл: Бизнес: 29597 (168 төлбөр); Мэдээллийн технологи: 63493 (236 төлбөр); Хэл: 8685 (63 төлбөр).

In [14]:
pay = (fact_payment.merge(dim_date, on="date_key")
                   .merge(dim_course, on="course_key")
                   .merge(dim_branch, on="branch_key"))

mar = pay[pay["month"] == "2026-03"]
print("slice:", mar["amount"].sum(), len(mar))

by_city = pay.groupby("city")["amount"].sum().sort_values(ascending=False)
print(by_city.head(3).to_dict())

print(pay.groupby("field")["amount"].agg(count="size", total="sum"))

slice: 16463 78
{'Улаанбаатар': 46885, 'Дархан': 20830, 'Эрдэнэт': 14209}
                      count  total
field                             
Бизнес                  168  29597
Мэдээллийн технологи    236  63493
Хэл                      63   8685


Хариулт: 3 сар 16463 (78 төлбөр). Хамгийн их хот Улаанбаатар 46885. МТ 236 төлбөр 63493, Бизнес 168 төлбөр 29597, Хэл 63 төлбөр 8685.

## 10) Snowflake схем

Од схемд `dim_product` нь `category`-г давтан агуулдаг. Snowflake схемд category-г тусдаа `dim_category` болгоно: давхардал багасна, гэхдээ асуулгад нэмэлт JOIN хэрэгтэй.

### TASK 13: dim_product-ыг задлах

1. `dim_product`-оос `category`-г ялгаж `dim_category` (`category_key`, `category_name`) үүсгэ.
2. `dim_product`-д `category_key` гадаад түлхүүр нэм, `category` баганыг хас.
3. Snowflake-аар `category` -ээр орлогыг гарга. Од схемийн үр дүнтэй ижил байх ёстой.
4. Давхардал хэд хэмжээгээр буурсан бэ? Нэмэлт JOIN-ий сул тал юу вэ?

Хүлээгдэх хариу: 3 ангилал, 6 бүтээгдэхүүн. Орлого: Сургалтын хэрэгсэл 13300 (2026 он), нийт 2026 = 18970.

In [15]:
dim_category = d_pro[["category"]].drop_duplicates().reset_index(drop=True)
dim_category.insert(0, "category_key", range(1, len(dim_category) + 1))
dim_category = dim_category.rename(columns={"category": "category_name"})

dim_product_sf = (d_pro.merge(dim_category, left_on="category", right_on="category_name")
                       .drop(columns=["category", "category_name"]))
print(dim_category.shape, dim_product_sf.columns.tolist())

sf = (fact.merge(d_mon, on="month_key")
          .merge(dim_product_sf, on="product_key")
          .merge(dim_category, on="category_key"))
print(sf[sf["year"] == 2026].groupby("category_name")["revenue"].sum().to_dict())

star = d26.groupby("category")["revenue"].sum().to_dict()
print("од схемтэй тэнцэх үү:", sf[sf["year"] == 2026].groupby("category_name")["revenue"].sum().to_dict() == star)

(3, 2) ['product_key', 'product_name', 'unit_price', 'category_key']
{'Бичиг хэрэг': 1820, 'Сургалтын хэрэгсэл': 13300, 'Электрон': 3850}
од схемтэй тэнцэх үү: True


Хариулт: 3 ангилал, 6 бараа. Сургалтын хэрэгсэл 13300, од схемтэй ижил. Ангиллын нэр 6 удаа давтагдаж байсан бол одоо 3 болсон. Гэхдээ асуулга бүрт нэмэлт join хийх хэрэгтэй болно.

## 11) Fact constellation ба snapshot fact

`FactSales` (сар x бараа x салбар) ба `FactInventory` (өдөр x бараа x салбар) хоёр fact нь **ижил `dim_product`, `dim_branch`**-г хуваалцана. Fact бүр өөр grain, хэмжүүртэй.

`closing_stock` нь **snapshot** (тухайн өдрийн үлдэгдэл). Үүнийг хугацаагаар SUM хийх нь буруу, учир нь өдөр бүрийн үлдэгдэл нь өмнөх өдрүүдийн үлдэгдлийг давтан агуулна.

### TASK 14: Snapshot-ийг зөв нэгтгэх

1. `cube_fact_inventory.csv`-г уншиж, бүх мөрийн `closing_stock`-ыг SUM хий. Энэ утга ямар утгагүй вэ?
2. **Зөв** хэлбэр: 1-р сарын сүүлийн өдрийн (2026-01-31) нийт үлдэгдэл.
3. **Зөв** хэлбэр: өдөр бүрийн нийт үлдэгдлийн дундаж (өдрөөр SUM, дараа нь дундаж).
4. Хоёр fact-ийг `product_key`-ээр нэгтгэж (drill across) бараа бүрийн 2026-01 сарын борлуулсан `quantity` ба сарын эцсийн үлдэгдлийг хамт гарга.

Хүлээгдэх хариу: буруу SUM = 123048; зөв сарын эцсийн = 3539; өдрийн дундаж = 3969.3. 2026-01 сард борлуулсан нийт ширхэг = 167.

In [16]:
inv = pd.read_csv("cube_fact_inventory.csv")
print("буруу SUM:", inv["closing_stock"].sum())

last = inv[inv["date"] == "2026-01-31"]["closing_stock"].sum()
avg  = inv.groupby("date")["closing_stock"].sum().mean().round(1)
print("сарын эцсийн:", last, " өдрийн дундаж:", avg)

sales_jan = dw[dw["month"] == "2026-01"].groupby("product_key")["quantity"].sum().rename("sold")
end_stock = inv[inv["date"] == "2026-01-31"].groupby("product_key")["closing_stock"].sum().rename("end_stock")
cmp_ = pd.concat([sales_jan, end_stock], axis=1).join(d_pro.set_index("product_key")["product_name"])
print(cmp_)
print("нийт ширхэг:", sales_jan.sum())

буруу SUM: 123048
сарын эцсийн: 3539  өдрийн дундаж: 3969.3
             sold  end_stock product_name
product_key                              
1              42        387          Ном
2              73        695       Дэвтэр
3              14        743         Үзэг
4              12        519      Үүргэвч
5              11        545  Тооны машин
6              15        650       Хавтас
нийт ширхэг: 167


Хариулт: Буруу SUM 123048, өдөр бүрийн үлдэгдлийг нэмэхэд нэг бараа олон удаа тоологдож байгаа. Сарын эцсийнх 3539, өдрийн дундаж 3969.3. 1 сард 167 ширхэг зарагдсан.

## 12) Схем ба cube-ийн чанарын шалгалт

Лекцийн шалгах хүснэгт:

| Шалгах зүйл | Шалгуур |
|---|---|
| Grain | Нэг fact хүснэгтэд нэг түвшний мөр |
| Түлхүүр | Dimension PK цорын ганц, fact FK хүчинтэй |
| Давхардал | Хэмжээсийн хослол давтагдахгүй |
| Тулгалт | Эх өгөгдөл ба cube-ийн нийлбэр тэнцүү |
| Тооцоолол | AVG-д SUM ба COUNT, snapshot-д зөв дүрэм |

### TASK 15: Чанарын шалгалтын функц

Дараах функцийг бөглөж, `cube_fact_sales` (од схем) дээр ажиллуул. Бүх шалгалт `True` гарах ёстой. Дараа нь **тулгалтыг зориуд эвдэж** (fact-ээс 1 мөр хас) аль шалгалт `False` болохыг ажигла.

In [17]:
def quality_check(fact, dims, keys, grain, measure, expected_total):
    out = {}
    for dim, key in zip(dims, keys):
        out[f"PK давхцаагүй: {key}"] = dim[key].is_unique
        out[f"FK хүчинтэй: {key}"]  = fact[key].isin(dim[key]).all()
    out["grain давхцаагүй"] = not fact.duplicated(subset=grain).any()
    out["нийлбэр тэнцүү"]  = fact[measure].sum() == expected_total
    return out

res = quality_check(fact, [d_mon, d_pro, d_bra], ["month_key", "product_key", "branch_key"],
                    ["month_key", "product_key", "branch_key"], "revenue", 36449)
for k, v in res.items(): print(v, k)

broken = fact.drop(index=0)
print(quality_check(broken, [d_mon, d_pro, d_bra], ["month_key", "product_key", "branch_key"],
                    ["month_key", "product_key", "branch_key"], "revenue", 36449)["нийлбэр тэнцүү"])

True PK давхцаагүй: month_key
True FK хүчинтэй: month_key
True PK давхцаагүй: product_key
True FK хүчинтэй: product_key
True PK давхцаагүй: branch_key
True FK хүчинтэй: branch_key
True grain давхцаагүй
True нийлбэр тэнцүү
False


Хариулт: Бүгд True. 1 мөр хасахад зөвхөн нийлбэр тэнцүү False болсон, түлхүүрийн шалгалтууд өөрчлөгдөөгүй.

## Тайлангийн загвар

Нэр: Буяндэлгэр  Бүлэг: КУ5  Огноо: 2026.11.12

---

## 1. Cube ба cuboid
| Асуулт | Хариу |
|---|---|
| Cell-ийн тоо, нийт орлого (cube_small) | 8, 660 |
| УБ, Ном, Ном+УБ орлого | 440, 400, 270 |
| 3 хэмжээсийн cuboid-ийн тоо | 8 |
| Шатлалтай cube-ийн cuboid-ийн тоо | 48 |

---

## 2. OLAP үйлдлүүд
| Үйлдэл | Юу хийсэн | Үр дүн |
|---|---|---|
| Roll-up | 2026 оныг улирал, жилээр | 18970 |
| Drill-down | Q4-ийг сараар | 10 сар 1846 хамгийн их |
| Slice | 2026-01 сар | 1380, Үүргэвч 480 |
| Dice | 2 сар, Ном Дэвтэр, УБ Дархан | 8 мөр, 660 |
| Pivot | салбар ба барааг сольсон | нийлбэр 660 хэвээр |

---

## 3. Хэмжүүрийн төрөл
| Хэмжүүр | Төрөл | Зөв нэгтгэх арга |
|---|---|---|
| SUM, COUNT | Distributive | хэсгийн дүнг нэмнэ |
| AVG | Algebraic | SUM ба COUNT хадгалж хуваана |
| MEDIAN | Holistic | бүх өгөгдлөөс дахин бодно |
| closing_stock (snapshot) | Semi-additive | сарын эцсийн утга эсвэл өдрийн дундаж |

---

## 4. AOI
| Алхам | Хийсэн шийдвэр |
|---|---|
| Хасах атрибут ба шалтгаан | student_id, бүгд өөр утгатай |
| Ерөнхийлсөн шатлал | нас 20-29/30-39/40-49, мэргэжил МТ/Бизнес/Инженер |
| Үр дүнгийн мөр, хамгийн том бүлэг | 9 мөр, (20-29, МТ) 63 |
| Босго (мөрийн тоо) | 10, gpa нэмбэл 20 болж хэтэрнэ |

---

## 5. Схем
| Схем | Давуу тал | Сул тал |
|---|---|---|
| Star | энгийн, join цөөн | утга давтагдана |
| Snowflake | давхардал бага | join олон |
| Fact constellation | dimension хуваалцана | grain өөр байж болно |

Төлбөрийн схемийн grain, хэмжүүр, `COUNT(*)`-ийн утга учир: grain нь нэг төлбөр, хэмжүүр SUM(amount), COUNT(*) нь төлбөрийн тоо (467), оюутны тоо биш (160).

---

## 6. Дүгнэлт
Хоёулаа шатлал ашиглаж өгөгдлийг ерөнхий түвшинд нэгтгэдэг. Cube нь орлого гэх мэт тоон утгыг нэгтгэдэг бол AOI нь атрибутын утгыг ерөнхийлөөд count гаргадаг. Хоёуланд нь нийлбэр эх өгөгдөлтэй тэнцэх ёстой.

---

## 7. Reflection
1. Бүлгүүдийн мөрийн тоо өөр учир жижиг бүлэг том бүлэгтэй адил нөлөөлдөг.
2. Бүгд өөр утгатай учир үлдээвэл мөрүүд нийлэхгүй.
3. Snapshot fact-ийг хугацаагаар нэмж болохгүй гэдэг хэсэг.